# Lesson 24 · Mixture of Experts (MoE): a giant model, but each token wakes up only a few experts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/24-mixture-of-experts-en.ipynb)

This notebook accompanies [Lesson 24 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mixture-of-experts/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/24-mixture-of-experts.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
We write an MoE layer ourselves in numpy (a router, top-k, experts inside a block with a residual path), train it on
real MNIST, and recount the parameters of Mixtral 8x7B and DeepSeek-V3 from their config.json. Two questions:
**why does a model with 46.7 billion parameters use only 12.9 billion for each token**, and
**what happens when nothing forces the router to share out the work evenly?**

**What you'll do:**
1. Write an MoE layer: the router's softmax scores, top-k selection, gate normalization, adding the experts' outputs
   to the residual path; write the backpropagation by hand and check it with finite differences.
2. Count the total and active parameters of Mixtral 8x7B and DeepSeek-V3 from config.json, matching the real
   checkpoints down to the last parameter.
3. Train an MoE on MNIST, watch the router collapse without balancing, then fix it with Switch Transformer's
   auxiliary loss and DeepSeek-V3's selection-only bias.
4. Cap the capacity of each expert and count the dropped tokens.
5. Run the demo's int8 model on the 10,000 test images, see which expert gets which digit, and compare MoE with a
   dense network.

**Background:** [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/) (the
FFN layer and the residual stream) and
[Lesson 23 · Scaling laws](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/quy-luat-mo-rong/) (parameters and
compute); the backpropagation of [Lesson 06](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/), the
Adam optimizer of [Lesson 07](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/) and the residual path of
[Lesson 13](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/). **Run time:** under a minute on a CPU,
including the first download of MNIST (~11 MB) and of the demo's three small results files.

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. The full training
runs (33 configurations × 3 seeds × 10 epochs, a few minutes on 16 cores), however, are loaded from the very files
the demo uses; the notebook reruns a short version of each mechanism and checks what can be recomputed.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `fmt()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label):
# no balancing = orange, squares; with balancing = blue, circles; dense network = green, diamonds
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
SEQ = LinearSegmentedColormap.from_list("white_blue", ["#f7f9fc", BLUE, "#0b3a75"])  # 0 → large
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


def span(values, digits=2, scale=1.0):
    """The min–max range across seeds, as in the demo: 97.40–97.57 (a single number when both ends round the same)."""
    lo, hi = scale * min(values), scale * max(values)
    return fmt(lo, digits) if round(lo, digits) == round(hi, digits) else f"{fmt(lo, digits)}–{fmt(hi, digits)}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 4.1·10¹⁴."""
    e = math.floor(math.log10(abs(x)))
    return f"{fmt(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it
    is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(fmt(13007, 0), fmt(0.1, 1), fmt(-4.3805, 4), pct(0.993527), span([0.974, 0.9757], scale=100), sci(4.1e14))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. As in the demo's runs, we split them into:
- **training:** the first 55,000 images (every network learns on this part);
- **validation:** the last 5,000 images of the training file (the demo chooses the learning rate $\eta$ here);
- **test:** the 10,000 images of the test file, used only to report results.

The cell also loads three results files that the demo uses (`fetch_json`, with an MD5 check):
- `moe-counts.json`: the parameter counts of 10 real checkpoints, counted from the header of each safetensors file,
  part by part;
- `moe-mnist.json`: 33 MoE and dense-network configurations on MNIST × 3 seeds × 10 epochs (expert loads, dead
  experts, dropped tokens, the digit × expert matrix…);
- `moe-weights.json`: the MoE model that the demo runs in the browser (seed 0, int8 weights), with 64 test images.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Results of the long training runs (the same files as the demo on the web). When publishing, build.py replaces the
# placeholder string in the urlretrieve below with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
counts = fetch_json("moe-counts.json", "997b5403cce3c91f064a79fc2cacc3e4")["models"]
sweep = fetch_json("moe-mnist.json", "bf417d9bf0dfa1ec75f8a7d9f8e6e36c")
shipped = fetch_json("moe-weights.json", "891715ae39ce1d0e1aed71fa948b9786")
size = sum((Path("data") / f).stat().st_size for f in ["moe-counts.json", "moe-mnist.json", "moe-weights.json"])
print(
    f"training: {fmt(len(train_img), 0)} images · test: {fmt(len(test_img), 0)} images · the demo's 3 files: {fmt(size / 1e6, 2)} MB"
)

The pixels are divided by 255 and stored as float32, as in the demo's runs. The figure below shows the first 12 test
images; they are also the first 12 of the 64 images the demo lets you choose from (section 8).

In [ ]:
Xtr, ytr = (train_img[:55000] / 255.0).astype(np.float32), train_lab[:55000].astype(np.int64)
Xte, yte = (test_img / 255.0).astype(np.float32), test_lab.astype(np.int64)
BATCH = 128
print(
    f"training {fmt(len(Xtr), 0)} images ({len(Xtr) // BATCH} batches of 128 images per epoch) · test {fmt(len(Xte), 0)} images"
)

fig, axes = plt.subplots(1, 12, figsize=(10, 1.2))
for ax, img, lab in zip(axes, Xte, yte):
    ax.imshow(img.reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(str(lab), fontsize=10)
    ax.axis("off")
plt.show()

## 3 · An MoE layer from scratch

In each Transformer block (Lesson 17), attention is followed by an FFN layer: a small MLP that runs separately for
each token, and most of the model's parameters sit here. MoE replaces that FFN layer with $N$ FFN layers, called
**experts**, plus a **router**. For the row vector $h$ of one token:

$$s = \operatorname{softmax}(h W_r), \qquad \mathcal T = \operatorname{TopK}(s + b,\; k), \qquad
g_i = \frac{s_i}{\sum_{j \in \mathcal T} s_j}, \qquad u = h + \sum_{i \in \mathcal T} g_i\, \mathrm{FFN}_i(h)$$

- $W_r$ has $d \times N$ entries: the router is just **a linear layer** that scores the $N$ experts.
- $\mathcal T$ is the set of the $k$ highest-scoring experts. $b$ is DeepSeek-V3's load-balancing bias (section 6):
  it is used only to *select*, and never enters $g_i$; for now $b = 0$. It is not the bias $b$ in $xW + b$ of
  Lesson 01.
- $g_i$ is the **gate weight**, normalized over the $k$ selected experts (Mixtral). Switch Transformer ($k = 1$) keeps
  $g_i = s_i$ as is; with normalization, for $k = 1$ the gate would always be 1 and the router would learn nothing
  from the task.
- An expert that is not selected **does not run** for that token. $u$ adds the experts' outputs to the **residual
  path** $h$, like every FFN layer in a Transformer (and Lesson 13).

Our network is the demo's network (`reel/moe/mnist.py`): a 784-pixel image → the trunk $h = \mathrm{ReLU}(xW_0 + b_0)$
with 64 dimensions → an MoE layer with 8 experts 64 → 32 → 64, $\mathrm{FFN}_i(v) = \mathrm{ReLU}(vA_i + a_i)B_i + c_i$
→ an output layer of 10 logits. The cell below initializes the network exactly as the demo does and writes the
router.

In [ ]:
def init_moe(seed, E=8, H=32, d=64, d_in=784, n_cls=10, dtype=np.float32):
    """Initialized as in reel/moe/mnist.py: He for the layers before a ReLU, the experts' B at half scale,
    W_r ~ N(0, 0.02²)."""
    rng = np.random.default_rng(seed)

    def he(fan_in, shape):
        return (rng.standard_normal(shape) * math.sqrt(2.0 / fan_in)).astype(dtype)

    P = {"W0": he(d_in, (d_in, d)), "b0": np.zeros(d, dtype)}
    P["A"], P["a"] = he(d, (E, d, H)), np.zeros((E, H), dtype)
    P["B"], P["c"] = (rng.standard_normal((E, H, d)) * math.sqrt(1.0 / H) * 0.5).astype(dtype), np.zeros((E, d), dtype)
    P["Wo"] = (rng.standard_normal((d, n_cls)) * math.sqrt(1.0 / d)).astype(dtype)
    P["bo"] = np.zeros(n_cls, dtype)
    P["Wr"] = (rng.standard_normal((d, E)) * 0.02).astype(dtype)  # the router: no bias, as in Mixtral
    return P


def softmax(l):
    e = np.exp(l - l.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def route(h, Wr, k, bias=None, gate="renorm"):
    """Scores s = softmax(h W_r); select k experts by s + b (b only for selection); the gate g normalized ("renorm")
    or raw."""
    s = softmax(h @ Wr)
    key = s if bias is None else s + bias
    sel = np.argsort(-key, axis=1, kind="stable")[:, :k]  # k experts, best first; a tie goes to the smaller index
    sv = np.take_along_axis(s, sel, 1)
    g = sv / sv.sum(1, keepdims=True) if gate == "renorm" else sv
    return s, sel, g


def ffn(v, P, e):
    """Expert e: a small MLP d → H → d."""
    return np.maximum(v @ P["A"][e] + P["a"][e], 0) @ P["B"][e] + P["c"][e]


P0 = init_moe(seed=0)
n_par = {key: v.size for key, v in P0.items()}
print("number of entries:", {key: fmt(n, 0) for key, n in n_par.items()})
expert = n_par["A"] // 8 + n_par["a"] // 8 + n_par["B"] // 8 + n_par["c"] // 8
trunk = n_par["W0"] + n_par["b0"] + n_par["Wo"] + n_par["bo"]
print(f"one expert: {fmt(expert, 0)} parameters · trunk + output: {fmt(trunk, 0)} · router: {fmt(n_par['Wr'], 0)}")

One expert has $64 \cdot 32 + 32 + 32 \cdot 64 + 64$ parameters. The router is very small: one $64 \times 8$ matrix.

Let's go step by step with one token, here the first test image (a 7), on the **untrained** network. The cell below
prints the scores $s$ of the 8 experts, the two selected experts and their gates, then adds the outputs of the two
experts to the residual path with a loop.

In [ ]:
k = 2
x = Xte[:1]  # one row: one token
h = np.maximum(x @ P0["W0"] + P0["b0"], 0)
s, sel, g = route(h, P0["Wr"], k)
print("s =", " ".join(fmt(v, 3) for v in s[0]), f" (sum {fmt(s.sum(), 3)})")
print(f"top-{k}: experts {sel[0].tolist()}, gates g = {' and '.join(fmt(v, 3) for v in g[0])} (sum {fmt(g.sum(), 3)})")
u = h.copy()  # the residual path
for i, e in enumerate(sel[0]):
    u += g[0, i] * ffn(h, P0, e)  # only 2 of the 8 experts run
print(
    f"‖h‖ = {fmt(np.linalg.norm(h), 3)}, ‖u − h‖ = {fmt(np.linalg.norm(u - h), 3)}: what MoE adds to the residual path"
)

At initialization, $W_r$ is very small, so $hW_r \approx 0$ and every $s_i$ is close to $1/8 = 0.125$: the router
has no opinion yet, and the two experts are selected almost by chance. Section 5 shows that this very fact leads to
collapse.

Computing one token at a time is very slow. Real hardware does the opposite: it **gathers the tokens that selected the
same expert** (dispatch), runs that expert once on the whole group, then returns the results to their places
(combine). The `forward` function below does exactly that, with two additions used in sections 7 and 8: the capacity
limit `cf` (ignored for now) and `off` to switch an expert off entirely.
The last cell compares gathering by expert with the token-by-token loop on 12 images.

In [ ]:
def capacity_keep(sel, E, cf):
    """Which assignments fit within the expert's capacity C = ⌈cf·n·k/N⌉ (section 7). cf = None: no limit."""
    n, k = sel.shape
    if cf is None:
        return np.ones_like(sel, bool)
    C = math.ceil(cf * n * k / E)
    flat = sel.T.ravel()  # assignment j·n + t = token t's j-th choice: every first choice queues up first
    onehot = np.zeros((n * k, E), np.int64)
    onehot[np.arange(n * k), flat] = 1
    rank = np.cumsum(onehot, 0)[np.arange(n * k), flat]  # position (from 1) in that expert's queue
    return (rank <= C).reshape(k, n).T


def forward(P, X, k, bias=None, gate="renorm", cf=None, off=None):
    """z = (h + Σ_{i∈𝒯} gᵢ FFNᵢ(h)) Wo + bo with h = ReLU(x W0 + b0). off[e] = True switches expert e off."""
    E = P["A"].shape[0]
    hp = X @ P["W0"] + P["b0"]
    h = np.maximum(hp, 0)
    s, sel, g = route(h, P["Wr"], k, bias, gate)
    kept = capacity_keep(sel, E, cf)
    if off is not None:
        kept &= ~off[sel]  # a switched-off expert's assignments are dropped, like assignments over capacity
    u = h.copy()  # the residual path
    cache = []
    for e in range(E):  # gather: every token that selected expert e (and still has room)
        rows, slot = np.nonzero((sel == e) & kept)
        q = h[rows] @ P["A"][e] + P["a"][e]
        r = np.maximum(q, 0)
        o = r @ P["B"][e] + P["c"][e]
        u[rows] += g[rows, slot][:, None] * o  # return: each token selects an expert at most once
        cache.append((rows, slot, q, r, o))
    z = u @ P["Wo"] + P["bo"]
    return z, {"hp": hp, "h": h, "s": s, "sel": sel, "g": g, "kept": kept, "u": u, "cache": cache}


X12 = Xte[:12]
z12, C12 = forward(P0, X12, k=2)
u_loop = C12["h"].copy()
for t in range(12):  # the token-by-token loop, as in the cell above
    for i, e in enumerate(C12["sel"][t]):
        u_loop[t] += C12["g"][t, i] * ffn(C12["h"][t : t + 1], P0, e)[0]
assert np.allclose(u_loop, C12["u"], atol=1e-5)
print("both computations give the same u; the experts selected for the 12 images:", C12["sel"].tolist())
load12 = np.bincount(C12["sel"].ravel(), minlength=8)
print("assignments each expert receives:", load12.tolist(), f"(total {load12.sum()} = 12 tokens × k = 2)")

Each token goes through its own pair of experts, and each expert receives a different number of tokens: that is its
**load**. Each token takes up $k$ **assignments** (slots), so the total load is $12 \cdot 2$.

One detail of Mixtral: the paper writes the gate as $\operatorname{softmax}(\operatorname{TopK}(hW_r))$, that is, a
softmax over the $k$ largest logits only. The way we write it (a softmax over all $N$, then dividing by the sum of the
$k$ selected scores) is the same thing, because the denominator $\sum_j e^{l_j}$ of the full softmax cancels out:

In [ ]:
l12 = C12["h"] @ P0["Wr"]
lt = np.take_along_axis(l12, C12["sel"], 1)  # the k largest logits of each token
gate_mixtral = softmax(lt)
print(f"largest |our gate − softmax(TopK(logits))|: {sci(np.abs(C12['g'] - gate_mixtral).max())} (float32)")
assert np.allclose(C12["g"], gate_mixtral, atol=1e-6)

### Backpropagation through the router

The *selection* of $\mathcal T$ is not differentiable: change $W_r$ a little, and the selected set either stays the
same or jumps. The gradient reaches the router only through the **gates** $g_i$:
$\partial \mathcal L/\partial g_i = \delta_u \cdot \mathrm{FFN}_i(h)$, with
$\delta_u = \partial \mathcal L / \partial u$. With the normalized gate $g_i = s_i/S$, $S = \sum_{j\in\mathcal T} s_j$:

$$\frac{\partial \mathcal L}{\partial s_i} = \frac{1}{S}\Big(\frac{\partial \mathcal L}{\partial g_i}
- \sum_{j \in \mathcal T} g_j \frac{\partial \mathcal L}{\partial g_j}\Big) \;\; (i \in \mathcal T), \qquad
\frac{\partial \mathcal L}{\partial s_i} = 0 \;\; (i \notin \mathcal T)$$

then through the softmax as in Lesson 04. An expert that is not selected receives no gradient at all, not even
through the router: it learns only when some token selects it. `loss_grad` also takes the auxiliary loss of section 6
(`alpha`); for now `alpha = 0`.

In [ ]:
def loss_grad(P, X, y, k, bias=None, gate="renorm", cf=None, alpha=0.0):
    """Cross-entropy (+ α·auxiliary loss) and its gradient; returns ce, aux, G, each expert's load, the number of
    dropped assignments."""
    E, n = P["A"].shape[0], len(X)
    z, C = forward(P, X, k, bias, gate, cf)
    p = softmax(z)
    ce = float(-np.log(p[np.arange(n), y] + 1e-30).mean())
    dz = p.copy()
    dz[np.arange(n), y] -= 1
    dz /= n
    G = {"Wo": C["u"].T @ dz, "bo": dz.sum(0)}
    du = dz @ P["Wo"].T
    dh = du.copy()  # the residual path
    for key in ("A", "a", "B", "c"):
        G[key] = np.zeros_like(P[key])
    dg = np.zeros_like(C["g"])
    for e, (rows, slot, q, r, o) in enumerate(C["cache"]):
        if len(rows) == 0:
            continue  # an expert nobody selected: zero gradient
        dd = du[rows]
        dg[rows, slot] = (dd * o).sum(1)  # ∂L/∂gᵢ = δu · FFNᵢ(h)
        do = C["g"][rows, slot][:, None] * dd
        G["B"][e], G["c"][e] = r.T @ do, do.sum(0)
        dq = (do @ P["B"][e].T) * (q > 0)
        G["A"][e], G["a"][e] = C["h"][rows].T @ dq, dq.sum(0)
        dh[rows] += dq @ P["A"][e].T
    s, sel, g = C["s"], C["sel"], C["g"]
    sv = np.take_along_axis(s, sel, 1)
    dsv = (dg - (dg * g).sum(1, keepdims=True)) / sv.sum(1, keepdims=True) if gate == "renorm" else dg
    ds = np.zeros_like(s)
    np.put_along_axis(ds, sel, dsv, 1)
    aux = 0.0
    if alpha > 0:  # Switch's auxiliary loss (section 6): α·N·Σ fᵢ Pᵢ; f is not differentiable, P = the mean of s
        f = np.bincount(sel.ravel(), minlength=E) / (n * k)
        aux = float(alpha * E * (f * s.mean(0)).sum())
        ds += alpha * E * f[None] / n
    dl = s * (ds - (ds * s).sum(1, keepdims=True))  # through the softmax
    G["Wr"] = C["h"].T @ dl
    dh += dl @ P["Wr"].T
    dhp = dh * (C["hp"] > 0)
    G["W0"], G["b0"] = X.T @ dhp, dhp.sum(0)
    load = np.bincount(sel.ravel(), minlength=E)
    return ce, aux, G, load, int((~C["kept"]).sum())

A finite-difference check (Lesson 06) on a small network in float64: 5 inputs, 4 dimensions, 4 experts, 7 tokens,
with both the normalized gate and the raw gate, top-1 and top-2, with the auxiliary loss, with a capacity limit.
Every entry of every parameter is nudged by $\pm\varepsilon$, and the cell prints the largest relative error between
the two computations.

In [ ]:
def fd_check(P, G, *args, eps=1e-6):
    """The largest relative error between G and (L(θ + ε) − L(θ − ε)) / 2ε, with L = ce + aux of loss_grad(P, *args)."""

    def total():
        return sum(loss_grad(P, *args)[:2])

    worst = 0.0
    for key in P:
        for i in np.ndindex(P[key].shape):
            old = P[key][i]
            P[key][i] = old + eps
            lp = total()
            P[key][i] = old - eps
            lm = total()
            P[key][i] = old
            num = (lp - lm) / (2 * eps)
            worst = max(worst, abs(num - G[key][i]) / max(1e-4, abs(num) + abs(G[key][i])))
    return worst


rng = np.random.default_rng(0)
cases = [(2, "renorm", 0.3, None), (2, "raw", 0.3, None), (1, "raw", 0.3, None), (2, "renorm", 0.0, 0.6)]
worst_all = 0.0
for k_, gate_, alpha_, cf_ in cases:
    Pt = init_moe(seed=int(rng.integers(1000)), E=4, H=3, d=4, d_in=5, n_cls=3, dtype=np.float64)
    for key in ("a", "c", "b0"):
        Pt[key] = rng.standard_normal(Pt[key].shape) * 0.1
    Pt["Wr"] = rng.standard_normal(Pt["Wr"].shape)
    Xt, yt_ = rng.standard_normal((7, 5)), rng.integers(0, 3, 7)
    b_ = rng.standard_normal(4) * 0.05  # an arbitrary selection bias
    args = (Xt, yt_, k_, b_, gate_, cf_, alpha_)
    _, _, Gt, _, dropped = loss_grad(Pt, *args)
    w = fd_check(Pt, Gt, *args)
    worst_all = max(worst_all, w)
    print(
        f"top-{k_}, gate {gate_:6s}, α = {fmt(alpha_, 1)}, cf = {fmt(cf_, 1) if cf_ else '∞'}: largest relative error {sci(w)}"
        + (f" ({dropped} assignments dropped)" if cf_ else "")
    )
assert worst_all < 1e-5
print(f"largest over the 4 cases: {sci(worst_all)}")

Every gradient matches the finite differences (relative error below $10^{-5}$), even when assignments are dropped
by the capacity limit: a dropped assignment is simply not in the sum, so it receives no gradient either.

## 4 · Total and active parameters

Before training, let's count. An MoE model has two numbers: the **total parameters** (which have to sit in memory)
and the **active parameters** (which each token actually uses). We compute both for Mixtral 8x7B from its
config.json alone, and compare them with the counts from the header of each safetensors file of the real checkpoint
(`reel/moe/counts.py`, stored in `moe-counts.json`).

### 🤔 Predict first
Mixtral 8x7B has 8 experts per layer, and the name suggests "8 times a 7-billion model". What is its total number of
parameters? Each token uses 2 experts: how many times as much does that cost as using only 1?

In [ ]:
# config.json of mistralai/Mixtral-8x7B-v0.1 at commit fc7ac94 (only the fields the formula needs)
MIXTRAL = {
    "hidden_size": 4096,  # d
    "num_hidden_layers": 32,  # L
    "num_attention_heads": 32,
    "num_key_value_heads": 8,  # grouped-query attention: 8 K/V heads for 32 Q heads
    "vocab_size": 32000,
    "intermediate_size": 14336,  # the hidden width of each expert (SwiGLU)
    "num_local_experts": 8,  # N
    "num_experts_per_tok": 2,  # k
    "tie_word_embeddings": False,
}
MISTRAL_7B = {key: v for key, v in MIXTRAL.items() if key not in ("num_local_experts", "num_experts_per_tok")}


def llama_style_params(cfg, k=None):
    """The parameters, part by part, of a Llama/Mistral/Mixtral-style model (no biases), from config.json."""
    d, L, V = cfg["hidden_size"], cfg["num_hidden_layers"], cfg["vocab_size"]
    h, kv = cfg["num_attention_heads"], cfg["num_key_value_heads"]
    hd = d // h  # the dimension of one head
    E = cfg.get("num_local_experts", 1)
    k = k or cfg.get("num_experts_per_tok", 1)
    ffn = 3 * d * cfg["intermediate_size"]  # SwiGLU: three matrices, gate, up, down
    parts = {
        "embedding": V * d,
        "output layer": 0 if cfg["tie_word_embeddings"] else V * d,
        "attention": L * (d * h * hd + 2 * d * kv * hd + h * hd * d),  # Q, K, V, O
        "normalization": L * 2 * d + d,  # 2 RMSNorms per block + 1 at the end
        "router": L * d * E if E > 1 else 0,  # W_r: d × N per layer
        "FFN / experts": L * E * ffn,
    }
    total = sum(parts.values())
    active = total - L * E * ffn + L * k * ffn  # every shared part + k of the N experts in each layer
    return parts, total, active


parts_mx, total_mx, active_mx = llama_style_params(MIXTRAL)
for name, n in parts_mx.items():
    print(f"{name:>17}: {fmt(n, 0):>15} ({pct(n / total_mx, 1)})")
print(
    f"total: {fmt(total_mx, 0)} ≈ {fmt(total_mx / 1e9, 1)} billion · active (top-2): {fmt(active_mx / 1e9, 1)} billion"
)
ref = counts["mixtral-8x7b"]  # counted from the 995 tensors in the safetensors headers of the checkpoint
assert (total_mx, active_mx) == (ref["total"], ref["active"]), "differs from the checkpoint: check the config again"
assert (round(total_mx / 1e9, 1), round(active_mx / 1e9, 1)) == (46.7, 12.9)
print(f"matches the checkpoint's {ref['tensors']} tensors down to the last parameter")

**46.7 billion, not 8 × 7 = 56 billion**: only the FFN layers are multiplied by 8, while attention, the embeddings
and normalization are shared. The experts make up almost the whole model, but each token runs only 2 of the 8 experts
in each layer. The Mixtral paper rounds these two numbers to 47B and 13B.

Now top-1 versus top-2:

In [ ]:
_, total_dense, _ = llama_style_params(MISTRAL_7B)
_, _, active_k1 = llama_style_params(MIXTRAL, k=1)
print(f"Mixtral with k = 1: {fmt(active_k1, 0)} parameters per token")
print(f"Mistral 7B (dense, same architecture): {fmt(total_dense, 0)}; difference {fmt(active_k1 - total_dense, 0)}")
print(f"top-2 / top-1 = {fmt(active_mx / active_k1, 2)} times")
assert active_k1 - total_dense == parts_mx["router"]  # exactly the W_r matrices
assert total_dense == counts["mistral-7b"]["total"]
assert round(active_mx / active_k1, 2) == 1.78

With $k = 1$, each Mixtral token uses exactly as many parameters as Mistral 7B plus the routing matrices: the two
models have the same architecture, and Mixtral has 8 copies of the FFN instead of 1. Top-2 costs **1.78 times** as
much as top-1, not 2 times, because the shared part (attention, embeddings) runs once whatever $k$ is.

The figure below draws three bars whose length is the number of parameters: Mistral 7B, the part of Mixtral that runs
for each token, and all of Mixtral. The gray part is the shared part; each outlined blue block is one expert (counted
across all 32 layers), and the hatched green block is the dense network's FFN.

In [ ]:
shared_mx = total_mx - parts_mx["FFN / experts"]
per_expert = parts_mx["FFN / experts"] // 8
fig, ax = plt.subplots(figsize=(9, 2.6))
rows = [("Mistral 7B (dense)", 0, 1), ("Mixtral, per token (k = 2)", 2, 0), ("Mixtral, total (in memory)", 8, 0)]
for y, (label, n_exp, dense) in enumerate(rows):
    ax.barh(y, shared_mx / 1e9, color=GREY, edgecolor="white", height=0.6)
    for j in range(n_exp):
        ax.barh(y, per_expert / 1e9, left=(shared_mx + j * per_expert) / 1e9, color=BLUE, edgecolor="white", height=0.6)
    if dense:
        ax.barh(y, per_expert / 1e9, left=shared_mx / 1e9, color=GREEN, hatch="//", edgecolor="white", height=0.6)
    end = shared_mx + max(n_exp, dense) * per_expert
    ax.text(end / 1e9 + 0.5, y, f"{fmt(end / 1e9, 1)} billion", va="center", fontsize=10)
ax.set_yticks(range(3), [r[0] for r in rows])
ax.invert_yaxis()  # read from top to bottom: dense → per token → total
ax.set(xlabel="parameters (billions)", xlim=(0, 53))
ax.set_title(
    f"Shared part {fmt(shared_mx / 1e9, 2)} billion (gray) + each expert {fmt(per_expert / 1e9, 2)} billion (blue)"
)
fmt_axes(fig)
plt.show()

### It saves computation, not memory

An expert that is not selected still has to sit in memory, because the next token may select it. In bf16 (2 bytes
per parameter), Mixtral's weights take up:

In [ ]:
GiB = 2**30
print(
    f"{fmt(total_mx, 0)} × 2 bytes = {fmt(2 * total_mx / GiB, 1)} GiB, for {fmt(active_mx / 1e9, 1)} billion parameters per token"
)
assert 2 * total_mx == counts["mixtral-8x7b"]["bytes"]  # the total size of the tensors in the safetensors files (bf16)


def touched(N, k, B):
    """The fraction of the experts that a batch of B tokens touches, if each token selects k of N uniformly and
    independently."""
    return 1 - (1 - k / N) ** B


for N_, k_, name in [(8, 2, "Mixtral"), (256, 8, "DeepSeek-V3")]:
    print(
        f"{name}, experts a batch of B tokens touches: "
        + ", ".join(f"B = {B} → {pct(touched(N_, k_, B), 0)}" for B in (1, 8, 64, 512))
    )

Mixtral 8x7B's checkpoint files weigh **87.0 GiB** (because $46.7 \cdot 10^9 \cdot 2$ bytes). And even though each
token reads only a few experts, a batch of a few dozen tokens already touches almost all of them: with uniform,
independent routing, a batch of 8 tokens touches 90% of Mixtral's experts. (Real routing is not independent, so the
real number is lower; this is only an illustration.) MoE reduces the computation per token, not the memory. Large
models have to split the experts across many GPUs and send tokens back and forth between them.

### DeepSeek-V3: 256 small experts and one shared expert

DeepSeek-V3 differs from Mixtral in three places: MLA-style attention (K/V compressed through a low rank), dense FFNs
in the first 3 layers, and in each MoE layer **256 small experts** (a hidden layer of 2,048 instead of 18,432), of
which 8 are selected, plus **1 shared expert** that always runs.
The formula is longer, but the idea is the same: active = every shared part + $k/N$ of the routed experts.

In [ ]:
# config.json of deepseek-ai/DeepSeek-V3 at commit e815299 (only the fields the formula needs)
DSV3 = {
    "hidden_size": 7168, "num_hidden_layers": 61, "num_attention_heads": 128, "vocab_size": 129280,
    "intermediate_size": 18432, "moe_intermediate_size": 2048, "first_k_dense_replace": 3,
    "n_routed_experts": 256, "n_shared_experts": 1, "num_experts_per_tok": 8,
    "q_lora_rank": 1536, "kv_lora_rank": 512, "qk_nope_head_dim": 128, "qk_rope_head_dim": 64, "v_head_dim": 128,
}  # fmt: skip


def deepseek_v3_params(c):
    d, L, h, V = c["hidden_size"], c["num_hidden_layers"], c["num_attention_heads"], c["vocab_size"]
    qr, kvr, vd = c["q_lora_rank"], c["kv_lora_rank"], c["v_head_dim"]
    qk = c["qk_nope_head_dim"] + c["qk_rope_head_dim"]  # Q/K dimension of one head: regular part + RoPE part
    nd, E, k = c["first_k_dense_replace"], c["n_routed_experts"], c["num_experts_per_tok"]
    expert = 3 * d * c["moe_intermediate_size"]  # one small SwiGLU expert
    mla = d * qr + qr * h * qk + d * (kvr + c["qk_rope_head_dim"]) + kvr * h * (c["qk_nope_head_dim"] + vd) + h * vd * d
    parts = {
        "embedding + output layer": 2 * V * d,
        "attention (MLA)": L * mla,
        "normalization": L * 2 * d + d + L * (qr + kvr),
        "3 dense FFNs": nd * 3 * d * c["intermediate_size"],
        "shared experts": (L - nd) * c["n_shared_experts"] * expert,
        "router (W_r and bias b)": (L - nd) * (d * E + E),
        "routed experts": (L - nd) * E * expert,
    }
    total = sum(parts.values())
    return parts, total, total - (L - nd) * (E - k) * expert


parts_ds, total_ds, active_ds = deepseek_v3_params(DSV3)
for name, n in parts_ds.items():
    print(f"{name:>32}: {fmt(n, 0):>17} ({pct(n / total_ds, 1)})")
print(
    f"total {fmt(total_ds / 1e9, 1)} billion · active {fmt(active_ds / 1e9, 1)} billion ({pct(active_ds / total_ds, 1)})"
)
ds = counts["deepseek-v3"]
assert (total_ds, active_ds) == (ds["total"], ds["active"]), "differs from the checkpoint: check the config again"
assert round(total_ds / 1e9) == 671 and 37e9 <= active_ds < 38e9
mtp = ds["parts"]["mtp"]  # multi-token prediction (MTP) layer, from the header; unused in ordinary generation
print(
    f"the Hugging Face repo also holds an MTP module of {fmt(mtp / 1e9, 2)} billion: {fmt((total_ds + mtp) / 1e9, 2)} billion in all"
)
print(f"checkpoint files (FP8, with scale factors): {fmt(ds['bytes'] / GiB, 1)} GiB")
print(
    f"hidden layer: dense FFN {fmt(DSV3['intermediate_size'], 0)}, each expert {fmt(DSV3['moe_intermediate_size'], 0)}"
)

DeepSeek-V3 uses **37.6 billion of its 671 billion** parameters for each token, about 5.6%: the routed experts make
up almost the whole model, while each token runs only 8 of the 256 experts in each layer. The "685B" on the Hugging
Face page is 671 billion plus the MTP (multi-token prediction) module, which the model card lists as "14B". The
router has an extra 256 biases $b_i$ per layer: they are exactly the load-balancing biases of section 6, stored in
the checkpoint under the name `e_score_correction_bias`. Even stored in FP8 (1 byte per parameter), the files still
weigh 641 GiB, so DeepSeek-V3 has to run on many machines; the technical report limits each token to being sent to
at most 4 machines (nodes), to reduce network traffic.

The table below is read directly from `moe-counts.json` (not recomputed): 10 checkpoints, every number matched by
`counts.py` down to the last parameter between the config.json formula and the safetensors headers.

In [ ]:
print(f"{'model':>16} {'N / k':>8} {'total (B)':>10} {'active':>10} {'ratio':>7} {'checkpoint':>11}")
for mid, m in counts.items():
    nk = f"{m['E']} / {m['k']}" if m["E"] > 1 else "dense"
    print(
        f"{mid:>16} {nk:>8} {fmt(m['total'] / 1e9, 1):>10} {fmt(m['active'] / 1e9, 2):>10} "
        f"{pct(m['active'] / m['total'], 1):>7} {fmt(m['bytes'] / GiB, 1):>7} GiB"
    )
go, sc = counts["gpt-oss-120b"], counts["llama4-scout"]
print(f"gpt-oss-120b, active without the embeddings: {fmt(go['activeNoEmbed'] / 1e9, 2)} billion")
print(f"Llama 4 Scout, text part + vision tower: {fmt((sc['total'] + sc['parts']['vision']) / 1e9, 2)} billion")

Two conventions to know when reading model cards: gpt-oss calls 5.13 billion "active" because it doesn't count the
embedding matrix (just a table lookup) but does count the output layer; Llama 4 Scout lists 109 billion because it
adds the vision tower. The active ratio falls from 27.6% (Mixtral, 2 of 8) to 5.6% (DeepSeek-V3, 8 of 256) and 4.9%
(gpt-oss-120b, 4 of 128): many recent models use many small experts and select a small fraction of them, but not
all of them do (Llama 4 Scout: 16 experts, top-1, 15.9%).

## 5 · Training on MNIST: the router collapses

Now we train the network of section 3 on MNIST, with exactly the demo's recipe: Adam ($\beta_1 = 0.9$,
$\beta_2 = 0.999$), batches of 128 images, a constant $\eta$, the image order from `default_rng(1000 + seed)`. The
demo trains for 10 epochs (4,290 steps) with 3 seeds; here we run 1 epoch (429 steps), and every 43 steps we record
each expert's **load**: the share of the assignments it received during those 43 steps. An expert that receives less
than ¼ of its fair share ($\tfrac14 \cdot \tfrac18$ of the assignments) is called **dead**, as in the demo.

In [ ]:
def train(seed, k=1, gate="raw", lr=3e-3, alpha=0.0, gamma=0.0, epochs=1, E=8, every=43):
    """Adam as in reel/moe/mnist.py. Returns the network, the bias b (if γ > 0) and each expert's load at steps 1, 43,
    86…"""
    P = init_moe(seed, E)
    b1, b2 = 0.9, 0.999
    m = {key: np.zeros_like(w) for key, w in P.items()}
    v = {key: np.zeros_like(w) for key, w in P.items()}
    bias = np.zeros(E, np.float32) if gamma > 0 else None
    order = np.random.default_rng(1000 + seed)
    hist, acc_load, step = [], np.zeros(E), 0
    for _ in range(epochs):
        perm = order.permutation(len(Xtr))
        for b in range(len(Xtr) // BATCH):
            idx = perm[b * BATCH : (b + 1) * BATCH]
            _, _, G, load, _ = loss_grad(P, Xtr[idx], ytr[idx], k, bias, gate, None, alpha)
            step += 1
            lr_t = lr * math.sqrt(1 - b2**step) / (1 - b1**step)
            for key in P:
                m[key] = b1 * m[key] + (1 - b1) * G[key]
                v[key] = b2 * v[key] + (1 - b2) * G[key] ** 2
                P[key] -= (lr_t * m[key] / (np.sqrt(v[key]) + 1e-8)).astype(np.float32)
            if bias is not None:  # DeepSeek-V3 (section 6): raise b when underloaded, lower it when overloaded
                bias += np.float32(gamma) * np.sign(load.mean() - load).astype(np.float32)
            acc_load += load
            if step % every == 0 or step == 1:
                hist.append(acc_load / acc_load.sum())
                acc_load = np.zeros(E)
    return P, bias, np.array(hist)


def dead(share, E=8):
    return int((share < 0.25 / E).sum())


def stored_hist(name, seed):
    """The load every 43 steps of a 10-epoch run in moe-mnist.json (in thousandths, uint16)."""
    raw = base64.b64decode(sweep["runs"][name]["seeds"][seed]["hist"])
    return np.frombuffer(raw, np.uint16).reshape(-1, 8) / 1000


R = sweep["runs"]
print(
    "η chosen on the validation set (seed 0):",
    ", ".join(f"{n} {fmt(R[n]['lr'], 3)}" for n in ("e8k1_none", "e8k1_aux", "e8k2_bias_sm")),
)
print(
    f"demo: {sweep['setup']['epochs']} epochs = {fmt(sweep['setup']['steps'], 0)} steps; load recorded every {sweep['setup']['every']} steps"
)

### 🤔 Predict first
At the start, every $s_i$ is close to $0.125$ (section 3). We train top-1 (Switch-style, gate $g = s_i$), with
**nothing forcing an even split**. After the first 43 steps ($43 \cdot 128$ images), how many of the 8 experts still
receive work?

In [ ]:
live = {}
for seed in range(3):
    live[("e8k1_none", seed)] = train(seed, k=1, gate="raw", lr=R["e8k1_none"]["lr"])
    h_live = live[("e8k1_none", seed)][2]
    h_ref = stored_hist("e8k1_none", seed)[: len(h_live)]
    print(
        f"seed {seed}: load at step 1 = [{' '.join(fmt(v, 3) for v in h_live[0])}]"
        f" · first 43 steps = [{' '.join(fmt(v, 3) for v in h_live[1])}] → {dead(h_live[1])} dead experts"
        f" · end of epoch 1: {dead(h_live[-1])} dead; largest deviation from the demo's run {fmt(np.abs(h_live - h_ref).max(), 3)}"
    )
    assert abs(dead(h_live[1]) - dead(h_ref[1])) <= 1, "differs from the demo: check the data again"
    assert np.sort(h_live[0])[-2:].sum() > 0.5, "at step 1, the two busiest experts get more than half the images"
dead43 = [dead(live[("e8k1_none", s)][2][1]) for s in range(3)]
print(f"dead after 43 steps: {dead43} (demo: {[dead(stored_hist('e8k1_none', s)[1]) for s in range(3)]})")
assert min(dead43) >= 5

After just 43 steps, **5–7 of the 8 experts are already dead**: one or two experts receive almost all the tokens.
There is no bug in the code. At the start all the scores are nearly equal, but the load is already uneven in the very
first batch, before the network has learned anything: the highest score of most images falls on the same few
experts, and the two busiest experts get more than half the images (the "load at step 1" lines above). Then some
expert happens to reduce the loss a little more for a few tokens. The gradient through the gate $g_i = s_i$ pushes
its $s_i$ up, so it gets selected more often. Only selected experts receive gradients (section 3), so it learns
faster, and gets selected even more. The other experts learn nothing, so they never become attractive again.
Shazeer et al. (2017) described exactly this loop: *"the favored experts are trained more rapidly and thus are
selected even more by the gating network"*.

The figure below reads the loads of the demo's 10-epoch runs (seed 0): each row is an expert, each column is 43
steps, and a dark color means a large share of the assignments. On the left is top-1 without balancing; on the
right, the same network with the auxiliary loss of section 6.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 2.9), sharey=True)
for ax, name, title in [(axes[0], "e8k1_none", "no balancing"), (axes[1], "e8k1_aux", "auxiliary loss α = 0.01")]:
    hs = stored_hist(name, 0)
    im = ax.imshow(hs.T, aspect="auto", cmap=SEQ, vmin=0, vmax=1, interpolation="nearest",
                   extent=(0, len(hs) * 43, 7.5, -0.5))  # fmt: skip
    ax.set(xlabel="training step")
    ax.set_title(f"top-1, {title}\n{dead(hs[-1])} dead experts in the last 43 steps", fontsize=10)
axes[0].set_ylabel("expert")
axes[0].set_yticks(range(8), [str(e) for e in range(8)])
fig.colorbar(im, ax=axes, fraction=0.03, label="share of assignments")
fmt_axes(fig)
plt.show()
for name in ("e8k1_none", "e8k1_aux"):
    sd = R[name]["seeds"]
    print(
        f"{name}, 10 epochs, 3 seeds: dead {[q['dead'] for q in sd]}, "
        f"the busiest expert receives {span([q['maxload'] / 8 for q in sd], 0, 100)}% of the assignments on the test set"
    )

On the left, two or three rows stay dark blue for all 4,290 steps and the rest are almost white: the collapse happens
right at the start and never recovers. At the end, the busiest expert receives 49–100% of the assignments (100% for
seed 2: the whole model is down to a single expert). On the right, every row is light blue: each expert receives
about 1/8 of the assignments.

## 6 · Load balancing

### Switch Transformer's auxiliary loss

Fedus, Zoph and Shazeer (2021) add to the loss a small term that is smallest when the load is split evenly:

$$\mathcal L_{\text{bal}} = \alpha N \sum_{i=1}^{N} f_i P_i$$

- $f_i$: the share of the batch's assignments that went to expert $i$. Switch has $k = 1$, so that is the fraction of
  tokens that select $i$; with $k = 2$, we count both choices, as the demo does. $f_i$ is a count: **not
  differentiable**.
- $P_i$: the average score $\bar s_i$ the router assigns to $i$ over the batch: **differentiable**, so the gradient
  flows through here.
- $\alpha = 0.01$ in Switch. Multiplying by $N$ makes the term always equal to $\alpha$ when the load is split evenly,
  however many experts there are.

The gradient with respect to $s_i$ is $\alpha N f_i / n$: the busier an expert, the harder its score is pulled down.
The cell below computes $\mathcal L_{\text{bal}}$ for three hypothetical batches.

In [ ]:
N_, alpha = 8, 0.01
for name, f, P_ in [
    ("even split", np.full(N_, 1 / N_), np.full(N_, 1 / N_)),
    (
        "half to expert 0",
        np.r_[0.5, np.full(N_ - 1, 0.5 / (N_ - 1))],
        np.r_[0.5, np.full(N_ - 1, 0.5 / (N_ - 1))],
    ),
    ("everything to expert 0", np.eye(N_)[0], np.eye(N_)[0]),
]:
    print(f"{name:>25}: L_bal = {fmt(alpha * N_ * (f * P_).sum(), 4)}")

An even split gives exactly $\alpha$; putting everything on one expert gives $\alpha N$, 8 times as much. Retraining
top-1 with $\alpha = 0.01$ (same seeds, same $\eta$):

In [ ]:
for seed in range(3):
    live[("e8k1_aux", seed)] = train(seed, k=1, gate="raw", lr=R["e8k1_aux"]["lr"], alpha=0.01)
for name in ("e8k1_none", "e8k1_aux"):
    ends = [dead(live[(name, s)][2][-1]) for s in range(3)]
    tops = [live[(name, s)][2][-1].max() * 8 for s in range(3)]
    print(f"{name}, 1 epoch: dead in the last 43 steps {ends}; largest load {span(tops, 2)} times the fair share")
end_none = [dead(live[("e8k1_none", s)][2][-1]) for s in range(3)]
end_aux = [dead(live[("e8k1_aux", s)][2][-1]) for s in range(3)]
assert max(end_aux) < min(end_none)

The auxiliary loss saves the experts: at the end of the epoch 0–2 experts are dead instead of 5–7, and the busiest
expert receives only about 2 times its fair share (after the demo's 10 epochs: no expert is dead). The price is a
second objective that pulls the router away from the main task. With too large an $\alpha$, the router splits the
work evenly rather than by content (section 9 measures that).

### DeepSeek-V3's selection-only bias

Wang et al. (2024), and then DeepSeek-V3, drop the auxiliary loss and give each expert a bias $b_i$ **used only when
selecting**: $\mathcal T = \operatorname{TopK}(s + b, k)$, while the gates are still computed from $s$. After each
step, based on the load of the whole batch:

$$b_i \leftarrow b_i + \gamma \cdot \operatorname{sign}(\bar c - c_i)$$

where $c_i$ is the number of assignments expert $i$ has just received and $\bar c$ is the average: an underloaded
expert's $b_i$ goes up, an overloaded expert's goes down. DeepSeek-V3 uses $\gamma = 0.001$ (and $\gamma = 0$ for the
last 500 billion tokens). $b$ is not in the loss function, so it doesn't pull the router away from the main task. The
cell below checks that: nudge $b$ slightly without changing any selection, and the loss stays the same down to the
last bit.

In [ ]:
Pt = init_moe(seed=3, E=4, H=3, d=4, d_in=5, n_cls=3, dtype=np.float64)
Pt["Wr"] = rng.standard_normal(Pt["Wr"].shape)
Xt, yt_ = rng.standard_normal((7, 5)), rng.integers(0, 3, 7)
b_ = rng.standard_normal(4) * 0.05
base_loss = loss_grad(Pt, Xt, yt_, 2, b_)[0]
for i in range(4):
    b2 = b_.copy()
    b2[i] += 1e-9
    assert loss_grad(Pt, Xt, yt_, 2, b2)[0] == base_loss
print(f"the loss {fmt(base_loss, 6)} stays the same when each bᵢ is nudged: the bias receives no gradient at all")

We train top-2 (normalized gate, as in Mixtral) without balancing and with the bias at $\gamma = 0.001$ (this is
exactly the model the demo runs in the browser, section 8), 3 seeds per configuration, then measure the load on the
10,000 test images:

In [ ]:
for seed in range(3):
    live[("e8k2_none", seed)] = train(seed, k=2, gate="renorm", lr=R["e8k2_none"]["lr"])
    live[("e8k2_bias_sm", seed)] = train(seed, k=2, gate="renorm", lr=R["e8k2_bias_sm"]["lr"], gamma=1e-3)


def test_share(name, seed):
    """The share of the assignments on the test set, and the accuracy, of a freshly trained model."""
    P, bias, _ = live[(name, seed)]
    k = 1 if "k1" in name else 2
    z, C = forward(P, Xte, k, bias, "raw" if k == 1 else "renorm")
    return np.bincount(C["sel"].ravel(), minlength=8) / C["sel"].size, (z.argmax(1) == yte).mean()


share = {key: test_share(*key) for key in live}
for name in ("e8k1_none", "e8k1_aux", "e8k2_none", "e8k2_bias_sm"):
    sh = [share[(name, s)][0] for s in range(3)]
    print(
        f"{name:>13}, 1 epoch: dead {[dead(q) for q in sh]}, largest load {span([q.max() * 8 for q in sh], 2)}, "
        f"{span([share[(name, s)][1] for s in range(3)], 2, 100)}% correct"
    )
assert all(share[("e8k2_bias_sm", s)][0].max() < share[("e8k2_none", s)][0].max() for s in range(3))
print("the bias b after 1 epoch (seed 0):", " ".join(fmt(v, 3) for v in live[("e8k2_bias_sm", 0)][1]))

Without balancing, top-2 collapses too: only 3 experts stay alive, and the busiest expert receives nearly 4 times its
fair share (with $k = 2$, the maximum is $N/k = 4$: that expert is in every pair). The bias $b$ keeps all 8 experts
alive, just by changing the choice of the tokens that are on the fence, without adding any term to the loss.

The accuracy after 1 epoch doesn't tell us anything yet (each configuration uses the $\eta$ the demo chose for 10
epochs). The demo's 10-epoch runs:

In [ ]:
def seeds(name, f):
    """One value per seed for a configuration in moe-mnist.json."""
    return [f(q) for q in R[name]["seeds"]]


for name in ("e8k2_none", "e8k2_aux", "e8k2_bias_sm"):
    print(
        f"{name:>13}, 10 epochs: {span(seeds(name, lambda q: q['test']), 2, 100)}% correct, "
        f"dead {seeds(name, lambda q: q['dead'])}, largest load {span(seeds(name, lambda q: q['maxload']), 2)}"
    )

Without a capacity limit, the three approaches are within seed-to-seed variation: the collapse doesn't lower the
accuracy here. The model has become a smaller dense network, with a few experts doing all the work, and the dead
experts only cost memory. The cost shows up when each expert can receive only a limited number of tokens.

## 7 · Capacity: tokens dropped when an expert is full

On real hardware, each expert processes a buffer of fixed size: tensors must have shapes known in advance, and the
experts usually sit on different GPUs. Switch Transformer gives each expert

$$C = \Big\lceil \mathrm{cf} \cdot \frac{n\,k}{N} \Big\rceil$$

slots in a batch of $n$ tokens, with a **capacity factor** cf; cf = 1 is exactly the fair share. A token that arrives
when the expert is already full is **dropped** at that layer: the expert doesn't run, and the token goes straight
through the residual path to the next layer. `capacity_keep` in section 3 queues up all of the batch's first choices
first, and only then the second choices, as the demo does.

### 🤔 Predict first
Suppose the router is perfectly balanced *on average*: each token selects 1 of the 8 experts uniformly at random and
independently. A batch of 128 tokens, cf = 1, so $C = 16$, exactly the average number of tokens each expert
receives. What percentage of the tokens is dropped?

In [ ]:
n_, N_ = 128, 8
sim_rng = np.random.default_rng(0)
for cf in (1.0, 1.25, 1.5, 2.0):
    C_ = math.ceil(cf * n_ / N_)
    # the number of tokens an expert receives ~ binomial(128, 1/8); the part over C is dropped
    over = sum(math.comb(n_, x) * (1 / N_) ** x * (1 - 1 / N_) ** (n_ - x) * (x - C_) for x in range(C_ + 1, n_ + 1))
    sim = np.mean([(~capacity_keep(sim_rng.integers(0, N_, (n_, 1)), N_, cf)).mean() for _ in range(2000)])
    print(
        f"cf = {fmt(cf, 2)}: C = {C_} → {pct(over * N_ / n_)} dropped (binomial) · simulating 2,000 batches: {pct(sim)}"
    )
    if cf == 1.0:
        drop_uniform = over * N_ / n_
        assert abs(sim - drop_uniform) < 0.005

About 9%, even though the *average* load is exactly equal to the capacity: by chance, one expert receives 20 tokens
and another only 12; the excess is dropped, and the shortfall leaves part of the buffer empty. That is why the
capacity factor is usually a little above 1 (Switch found cf 1.0–1.25 to work best), and the more skewed the load,
the more gets dropped. Measured on the models we just trained for 1 epoch, with batches of 128 test images as in the
demo:

In [ ]:
def eval_capacity(P, k, gate, bias, cf):
    """The accuracy and the share of dropped assignments on the 10,000 test images, in batches of 128 images."""
    hit = drop = 0
    for b in range(0, len(Xte), BATCH):
        z, C = forward(P, Xte[b : b + BATCH], k, bias, gate, cf)
        hit += int((z.argmax(1) == yte[b : b + BATCH]).sum())
        drop += int((~C["kept"]).sum())
    return hit / len(Xte), drop / (len(Xte) * k)


cap = {}
for name in ("e8k1_none", "e8k1_aux"):
    for seed in range(3):
        P, bias, _ = live[(name, seed)]
        cap[(name, seed)] = eval_capacity(P, 1, "raw", bias, 1.0)
    acc_, drop_ = zip(*(cap[(name, s)] for s in range(3)))
    print(
        f"{name}, 1 epoch, cf = 1: {span(drop_, 0, 100)}% of the assignments dropped, still {span(acc_, 1, 100)}% correct"
    )
assert all(cap[("e8k1_none", s)][1] > cap[("e8k1_aux", s)][1] for s in range(3))

The collapsed model drops most of its assignments: the busiest expert receives several times $C$, the excess is
dropped, and the dead experts' buffers stay empty. The model with the auxiliary loss drops much less. The demo's
10-epoch runs give the same picture, more clearly:

In [ ]:
cfs = sweep["setup"]["cfs"]  # the capacity factors measured; None = no limit
i1, i0 = cfs.index(1.0), cfs.index(0.0)
for name in ("e8k1_none", "e8k1_aux", "e8k2_none", "e8k2_bias_sm"):
    print(
        f"{name:>13}, 10 epochs, cf = 1: {span(seeds(name, lambda q: q['cf'][i1][1]), 0, 100)}% dropped, "
        f"{span(seeds(name, lambda q: q['cf'][i1][0]), 1, 100)}% correct "
        f"(no limit: {span(seeds(name, lambda q: q['test']), 1, 100)}%; "
        f"cf = 0: {span(seeds(name, lambda q: q['cf'][i0][0]), 1, 100)}%)"
    )
print(f"e8k1_aux: median raw gate sᵢ {span(seeds('e8k1_aux', lambda q: q['gate1']), 2)}")

At cf = 1, unbalanced top-1 drops **64–88%** of its assignments and is only **86.1–96.4%** correct; with the
auxiliary loss it drops **19–20%** and is still **97.1–97.3%** correct, losing almost nothing compared with no limit.
A dropped token doesn't disappear: it still continues along the residual path $u = h$.

But look at the cf = 0 column, where *every* assignment is dropped: the top-1 model with the auxiliary loss is still
97.2–97.3% correct. In this model the experts contribute almost nothing (its raw gate $s_i$ has a median of only
0.30–0.33), so it tolerates a small capacity mainly because it doesn't need the experts: the trunk and the residual
path are already enough for MNIST. The top-2 model with the bias does need them (cf = 0: 80.0–89.8%), and yet at
cf = 1 it drops only 8–10% of its assignments and is 97.4–97.5% correct.

How much does the residual path matter? The demo also trains a kind of MoE **without a residual path**, as in Jacobs
et al. (1991): each expert reads the 784 pixels itself and outputs 10 logits (784 → 8 → 10),
$z = \sum_{i\in\mathcal T} g_i\, \mathrm{FFN}_i(x)$. There, a dropped token loses that expert's contribution
altogether. The figure below plots the accuracy (left) and the share of dropped assignments (right) against the
capacity factor, top-2, one marker per seed: solid lines have a residual path, dashed lines don't; blue circles are
balanced (auxiliary loss α = 0.01), orange squares are unbalanced.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
x_cf = [c for c in cfs if c is not None]
series = [("e8k2_aux", BLUE, "o", "-"), ("e8k2_none", ORANGE, "s", "-"), ("h8k2_aux", BLUE, "o", "--"),
          ("h8k2_none", ORANGE, "s", "--")]  # fmt: skip
labels = {"e8k2_aux": "residual path, balanced", "e8k2_none": "residual path, unbalanced",
          "h8k2_aux": "no residual path, balanced", "h8k2_none": "no residual path, unbalanced"}  # fmt: skip
for name, color, marker, ls in series:
    vals = np.array([[q["cf"][i] for i in range(len(x_cf))] for q in R[name]["seeds"]])  # seed × cf × (correct, drop)
    for ax, j in ((a1, 0), (a2, 1)):
        ax.plot(x_cf, 100 * np.median(vals[:, :, j], 0), color=color, ls=ls, lw=1.5, label=labels[name])
        for sd in vals:
            ax.scatter(x_cf, 100 * sd[:, j], color=color, marker=marker, s=18, zorder=3,
                       facecolors="none" if ls == "--" else color)  # fmt: skip
a1.set(xlabel="capacity factor cf", ylabel="% correct on the 10,000 test images", ylim=(0, 100))
a2.set(xlabel="capacity factor cf", ylabel="% of assignments dropped", ylim=(0, 100))
a1.set_title("Accuracy (top-2)")
a2.set_title("Dropped assignments (top-2)")
a1.legend(frameon=False, fontsize=8, loc="lower right")
fmt_axes(fig)
plt.tight_layout()
plt.show()
for name in ("h8k2_none", "h8k2_aux"):
    print(
        f"{name}: cf = 0 → {span(seeds(name, lambda q: q['cf'][i0][0]), 1, 100)}% correct; "
        f"cf = 1 → {span(seeds(name, lambda q: q['cf'][i1][1]), 0, 100)}% dropped, "
        f"{span(seeds(name, lambda q: q['cf'][i1][0]), 1, 100)}% correct"
    )
print(
    f"with a residual path, cf = 0 (everything dropped): {span(seeds('e8k2_aux', lambda q: q['cf'][i0][0]), 1, 100)}% correct"
)

With a residual path, cf = 0 (the MoE layer dropped entirely) is still 73–88% correct, because the trunk and the
output layer still run. Without a residual path, cf = 0 is only 9.8% correct (random guessing), and at cf = 1 the
unbalanced model falls to 85–87%. In a Transformer, the residual path is what allows tokens to be dropped without
breaking the model. DeepSeek-V3 goes further: thanks to the bias $b$ balancing well enough, it drops no tokens at all,
neither in training nor at inference.

## 8 · The demo's model: int8, run on 10,000 images

The model the demo runs in the browser is seed 0 of the configuration with top-2, a normalized gate and the bias at
$\gamma = 0.001$, trained for 10 epochs. To keep the file small, each matrix is stored as **int8** with one float16
scale per output column (for $A$ and $B$: per expert), and the vectors are stored as float16. Unpacking:
$W \approx q \cdot s$, with $q \in \{-127, \dots, 127\}$. This is also the kind of quantization that gpt-oss and
DeepSeek-V3 use at a larger scale (MXFP4, FP8) so that the experts fit in memory.

In [ ]:
def b64(text, dtype):
    """Decode an array that the demo stores as base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(text), dtype)


def unpack(t):
    if "f16" in t:
        return b64(t["f16"], np.float16).astype(np.float64).reshape(t["shape"])
    q = b64(t["i8"], np.int8).reshape(t["shape"]).astype(np.float64)
    scale_shape = list(t["shape"])
    scale_shape[-2] = 1  # one scale per output column (and per expert, for A, B)
    return q * b64(t["scale"], np.float16).astype(np.float64).reshape(scale_shape)


Pd = {key: unpack(t) for key, t in shipped["P"].items() if key != "bias"}
bias_d = unpack(shipped["P"]["bias"])
cfg_d = shipped["cfg"]
n_bytes = sum(len(base64.b64decode(t.get("i8", t.get("f16", "")))) for t in shipped["P"].values())
print(
    f"configuration: {cfg_d['E']} experts {cfg_d['d']} → {cfg_d['H']} → {cfg_d['d']}, top-{cfg_d['k']}, "
    f"{cfg_d['score']}, gate {cfg_d['gate']}, γ = {fmt(cfg_d['gamma'], 3)}"
)
print(f"{fmt(sum(w.size for w in Pd.values()), 0)} parameters in {fmt(n_bytes / 1024, 1)} KB (not counting the scales)")
print("the learned bias b:", " ".join(fmt(v, 3) for v in bias_d))

z_d, C_d = forward(Pd, Xte.astype(np.float64), cfg_d["k"], bias_d, cfg_d["gate"])
acc_d = (z_d.argmax(1) == yte).mean()
z0_d, _ = forward(Pd, Xte.astype(np.float64), cfg_d["k"], bias_d, cfg_d["gate"], cf=0.0)
acc0_d = (z0_d.argmax(1) == yte).mean()
print(f"{pct(acc_d)} correct on the 10,000 test images · the residual path alone (cf = 0): {pct(acc0_d)}")
assert abs(acc_d - shipped["live"]["test"]) < 0.0005 and abs(acc0_d - shipped["live"]["residual"]) < 0.0005, (
    "differs from the demo"
)

x64 = b64(shipped["digits"], np.uint8).reshape(-1, 784)
assert np.array_equal(x64, test_img[:64]) and shipped["labels"] == test_lab[:64].tolist()
print(f"the demo's 64 images are the first 64 test images; {(z_d[:64].argmax(1) == yte[:64]).sum()}/64 correct")

The int8 model is 97.57% correct, the same number the demo shows. The figure below shows the first 8 images: under
each image are the two selected experts and their gates. The first gate is usually much larger than the second, but
rarely equal to 1.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(10, 1.9))
for t, ax in enumerate(axes):
    ax.imshow(Xte[t].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    (e1, e2), (g1, g2) = C_d["sel"][t], C_d["g"][t]
    ax.set_title(f"digit {yte[t]} → {z_d[t].argmax()}\nE{e1}: {fmt(g1, 2)}\nE{e2}: {fmt(g2, 2)}", fontsize=9)
    ax.axis("off")
plt.show()
print(f"median gates: first {fmt(np.median(C_d['g'][:, 0]), 2)}, second {fmt(np.median(C_d['g'][:, 1]), 2)}")
plain = np.sort(np.argsort(-C_d["s"], 1, kind="stable")[:, :2], 1)
print(f"the bias b changes the expert pair of {pct((plain != np.sort(C_d['sel'], 1)).any(1).mean(), 1)} of the images")

The bias $b$ changes the expert pair of about one fifth of the images: those are the undecided tokens that it pushes
toward underloaded experts. For most tokens, the choice is still the router's.

### Switching off one expert at a time

How important is one expert? Switch it off (all its assignments are dropped, like assignments over capacity; the
tokens are not rerouted to another expert) and measure again:

In [ ]:
acc_off = []
for e in range(8):
    z_off, _ = forward(Pd, Xte.astype(np.float64), 2, bias_d, "renorm", off=np.arange(8) == e)
    acc_off.append((z_off.argmax(1) == yte).mean())
print("expert 0…7 switched off:", " · ".join(pct(a) for a in acc_off))
assert np.allclose(acc_off, shipped["live"]["off"], atol=0.0005), "differs from the demo: check the data again"
print(f"the largest loss is from switching off expert {int(np.argmin(acc_off))}: {pct(min(acc_off))} left")
lost = [acc_d - a for a in acc_off]
print(f"a loss of {span(lost, 2, 100)} points from {pct(acc_d)} with all 8 experts")

No expert is indispensable: switching off any one of them costs only 0.29 to 2.32 points, because each token still
has its second expert and the residual path.

## 9 · What do the experts learn?

The name "expert" suggests a medical specialist: each expert takes care of one topic. On MNIST, the most obvious
"topic" is the digit.

### 🤔 Predict first
Count, for each digit, which expert is the router's first choice. There are 10 digits and 8 experts. Do you see "one
expert per digit"? About what percentage of an expert's images does its most common digit make up?

In [ ]:
mat = np.zeros((10, 8), np.int64)
np.add.at(mat, (yte, C_d["sel"][:, 0]), 1)  # rows: digits; columns: the first expert
purity = mat.max(0).sum() / mat.sum()  # the share of each expert's images that belong to its most common digit


def mutual_info(a, b):
    """I(a; b) in bits, from the joint count table."""
    joint = np.zeros((a.max() + 1, b.max() + 1))
    np.add.at(joint, (a, b), 1)
    joint /= joint.sum()
    pa, pb = joint.sum(1, keepdims=True), joint.sum(0, keepdims=True)
    nz = joint > 0
    return float((joint[nz] * np.log2(joint[nz] / (pa @ pb)[nz])).sum())


I_d = mutual_info(yte, C_d["sel"][:, 0])
rows_ = mat / np.linalg.norm(mat, axis=1, keepdims=True)
cos = rows_ @ rows_.T  # two digits are similar when they are sent to the same experts
pairs = sorted(((cos[a, b], a, b) for a in range(10) for b in range(a + 1, 10)), reverse=True)[:4]
print(f"purity: {pct(purity, 1)} · I(digit; expert) = {fmt(I_d, 2)} bits (log₂10 = {fmt(math.log2(10), 2)} bits)")
print("the 4 digit pairs with the most similar routing rows:", ", ".join(f"{a}–{b} ({fmt(c, 2)})" for c, a, b in pairs))
ref_mat = np.array(R["e8k2_bias_sm"]["seeds"][0]["mat"])  # the same model in float32, before the int8 compression
print(f"differs from the demo's float32 matrix on {np.abs(mat - ref_mat).sum()} of the 10,000 images")
assert abs(I_d - R["e8k2_bias_sm"]["seeds"][0]["I"]) < 0.01 and (pairs[0][1], pairs[0][2]) == (3, 5)

There is no "one expert per digit". The figure below draws that matrix, each row divided by the number of images of
that digit (each row adds up to 100%; a cell shows its number when it is 10% or more). A few digits almost always go
to one expert (0 to E7, 1 to E4, 9 to E5), but those same experts also receive other digits, and some digits (2, 5,
8) are split across several experts.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.6))
frac = mat / mat.sum(1, keepdims=True)
im = ax.imshow(100 * frac, cmap=SEQ, vmin=0, vmax=100)
for (r, c), v in np.ndenumerate(frac):
    if v >= 0.1:
        ax.text(c, r, f"{100 * v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 0.5 else "black")
ax.set_xticks(range(8), [f"E{e}" for e in range(8)])
ax.set_yticks(range(10), [str(d) for d in range(10)])
ax.set(xlabel="first selected expert", ylabel="digit")
ax.set_title("The demo's model: digit → expert (% of each row)", fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.04, label="% of the digit's images")
fmt_axes(fig)
plt.show()

Rows 3 and 5 are the most similar (both concentrated on E0 and E1), and E5 receives almost all the 9s along with half
of the 4s: the router groups digits that **look alike**, because it only sees the trunk's features $h$ and doesn't
know the labels. The routing still carries information about the digit (1.64 bits, nearly half of log₂10), but that
is information about the shape of the strokes.

One model could be a coincidence. The demo counts, over 4 balanced top-2 configurations × 3 seeds, the 4 digit pairs
with the most similar routing rows in each run:

In [ ]:
PAIR_RUNS = ["e8k2_bias_sm", "e8k2_aux", "e8k2_aux01", "e8k2_bias"]
pair_count, n_runs = {}, 0
for name in PAIR_RUNS:
    for q in R[name]["seeds"]:
        n_runs += 1
        for a, b, _ in q["pairs"]:
            pair_count[(a, b)] = pair_count.get((a, b), 0) + 1
top = sorted(pair_count.items(), key=lambda kv: -kv[1])[:3]
print(f"{n_runs} runs:", ", ".join(f"{a}–{b} in {n}/{n_runs}" for (a, b), n in top))
for name, label in [("e8k2_frozen", "random router, not learned"), ("e8k2_bias_sm", "bias γ = 0.001"),
                    ("e8k2_aux01", "auxiliary loss α = 0.1"), ("e8k2_aux1", "auxiliary loss α = 1")]:  # fmt: skip
    I_s = seeds(name, lambda q: q["I"])
    print(
        f"{label:>36}: I(digit; expert) = {span(I_s, 2)} bits, {span(seeds(name, lambda q: q['test']), 2, 100)}% correct"
    )

3 and 5 go together in 11/12 runs, 4 and 9 in 9/12: exactly the pairs that Lesson 01 found hard to separate with a
single neuron. Two more comparisons:
- A router that **doesn't learn** (a fixed random matrix) still carries 0.44–0.66 bits about the digit, because the
  trunk in front of it still learns, so different digits fall into different regions even of a random projection.
  Learning adds about 1 bit.
- **Too strong** an auxiliary loss ($\alpha = 1$) brings it down to 0.68–0.79 bits, versus 1.52–1.63 bits at
  $\alpha = 0.1$: the router is forced to split the work evenly rather than by content. That is why DeepSeek-V3
  switched to the bias $b$, keeping a very small auxiliary loss ($\alpha = 0.0001$) only to avoid extreme imbalance
  within individual sequences.

Real language models are no different. Mixtral measured routing on the domains of The Pile (ArXiv, PubMed,
PhilPapers…) and reports: *"we do not observe obvious patterns in the assignment of experts based on the topic"*.
Routing follows syntax: `self` in Python and the indentation tokens in code often go to the same expert, and two
consecutive tokens tend to choose the same expert. OLMoE (64 small experts, trained from scratch) found clearer domain
specialization on distinctive domains such as arXiv and GitHub, nearly uniform routing on general domains such as C4,
and strong specialization by token. "Expert" is a name, not a medical specialist.

## 10 · Does MoE beat a dense network?

On MNIST, the answer has to be measured, not guessed. The demo compares the MoE configurations with dense networks
that have the same trunk and one FFN with a hidden layer of $H$ in place of the 8 experts: $H = 64$ has the same
**number of active parameters** as top-2 MoE (two experts of 32), $H = 256$ has the same **total parameters** (eight
experts of 32). The figure below plots the test-set accuracy, one marker per seed: the dense networks are green
diamonds, MoE is blue circles; the labels give the number of parameters that run for each image, and the total.

In [ ]:
CMP = ["dense32", "dense64", "dense128", "dense256", "e8k2_aux", "e8k2_bias_sm", "e16k4_aux", "e32k8_aux",
       "e64k16_aux", "s1e15k3_aux", "e2k2_aux", "e4k2_aux", "e16k2_aux", "e32k2_aux"]  # fmt: skip


def label(name):
    c, p = R[name]["cfg"], R[name]["params"]
    if c["E"] == 1:
        return f"dense H {c['H']} · {fmt(p['total'], 0)}"
    kind = f"{c['E']} × H {c['H']} top-{c['k']}" + (" + 1 shared" if c["shared"] else "")
    kind += " · bias" if c["bal"] == "bias" else ""
    return f"{kind} · {fmt(p['active'], 0)} / {fmt(p['total'], 0)}"


fig, ax = plt.subplots(figsize=(9, 5.2))
for y, name in enumerate(CMP):
    is_dense = R[name]["cfg"]["E"] == 1
    acc_s = seeds(name, lambda q: 100 * q["test"])
    ax.scatter(acc_s, [y] * 3, marker="D" if is_dense else "o", color=GREEN if is_dense else BLUE, s=36, zorder=3)
    ax.axhline(y, color=GREY, lw=0.4, alpha=0.4, zorder=1)
ax.set_yticks(range(len(CMP)), [label(n) for n in CMP], fontsize=8.5)
ax.invert_yaxis()
ax.set(xlabel="% correct on the 10,000 test images (3 seeds)")
ax.set_title("With a residual path: MoE and dense networks (labels: active / total parameters)", fontsize=11)
fmt_axes(fig)
plt.tight_layout()
plt.show()
for name in ("dense64", "dense128", "dense256", "e8k2_aux", "e8k2_bias_sm"):
    print(f"{label(name):>40}: {span(seeds(name, lambda q: q['test']), 2, 100)}%")
moe_top2 = seeds("e8k2_aux", lambda q: q["test"]) + seeds("e8k2_bias_sm", lambda q: q["test"])
print(f"top-2 MoE (auxiliary loss and bias, 6 runs): {span(moe_top2, 2, 100)}%")

With a residual path, top-2 MoE is **97.40–97.57%** correct, better than the dense network with the same number of
active parameters ($H = 64$: 96.81–97.33%) and even the dense network with the same total parameters
($H = 256$: 97.15–97.38%), but only by a few tenths of a point, just outside the seed-to-seed variation. The dense
network with $H = 128$ (97.07–97.66%) overlaps the MoE range. Splitting the experts finer or adding a shared expert
doesn't make a clear difference at this size either. And **without a residual path**:

In [ ]:
for name, what in [("hd24", "single net 784 → 24 → 10 (same active)"), ("h8k2_aux", "MoE, 8 experts 784 → 8 → 10, top-2"),
                   ("hd64", "single net 784 → 64 → 10")]:  # fmt: skip
    p = R[name]["params"]
    print(
        f"{what:>40}: active {fmt(p['active'], 0):>6}, total {fmt(p['total'], 0):>6} parameters → "
        f"{span(seeds(name, lambda q: q['test']), 2, 100)}%"
    )


def choices(N, k):
    n = math.comb(N, k)
    return fmt(n, 0) if n < 1e6 else sci(n)


print(f"8 × H 32 top-2: {choices(8, 2)} combinations")
for name in ("e16k4_aux", "e32k8_aux", "e64k16_aux", "s1e15k3_aux", "e2k2_aux", "e4k2_aux", "e16k2_aux", "e32k2_aux"):
    c = R[name]["cfg"]
    print(
        f"{label(name):>40}: {span(seeds(name, lambda q: q['test']), 2, 100)}% · "
        f"combinations: {choices(c['E'], c['k'])} · dead {seeds(name, lambda q: q['dead'])}"
    )

Without a residual path, MoE beats the single network with the same active part, but **loses to a single network
with fewer total parameters** (784 → 64 → 10: 50,890 parameters versus 57,232). One explanation: each expert sees only
about $k/N$ = a quarter of the images. From 2 to 32 experts (top-2), the accuracy stays within seed-to-seed
variation, and with 32 experts some experts die even with the auxiliary loss.

Splitting the experts finer (DeepSeekMoE) gives more ways to combine them: choosing 2 of 8 gives 28 combinations, 4 of
16 gives 1,820, 16 of 64 about $4.9 \cdot 10^{14}$, but at MNIST's size this doesn't change the accuracy. The large
advantage of MoE that the papers report comes from very large data: Switch Transformer trains up to 7 times faster for
the same computation, on more than 180 billion tokens. MNIST is too small to see that; what it does show clearly is
the mechanism: collapse, balancing, capacity, and what the experts actually learn.

## 11 · Try changing things

The cells below rerun `train()` (1 epoch, seed 0) with other settings and measure on the 10,000 test images. You can
edit the values and run them again. One seed and one epoch only show a trend: add seeds before drawing conclusions.

In [ ]:
def summary(P, bias, k, gate):
    """The accuracy, dead experts, largest load and I(digit; expert) on the 10,000 test images."""
    z, C = forward(P, Xte, k, bias, gate)
    sh = np.bincount(C["sel"].ravel(), minlength=8) / C["sel"].size
    acc = (z.argmax(1) == yte).mean()
    return f"{pct(acc)} correct, {dead(sh)} dead, largest load {fmt(sh.max() * 8, 2)}, I = {fmt(mutual_info(yte, C['sel'][:, 0]), 2)} bits"


def trial(k=2, gate="renorm", lr=1e-3, alpha=0.0, gamma=0.0, seed=0):
    P, bias, _ = train(seed, k=k, gate=gate, lr=lr, alpha=alpha, gamma=gamma)
    return summary(P, bias, k, gate)


print("Auxiliary loss α (top-2):")
for a in (0.01, 0.1, 1.0):
    print(f"  α = {fmt(a, 2)}: {trial(alpha=a)}")
print("Bias update rate γ (top-2):")
for gm in (1e-4, 1e-3, 1e-2):
    print(f"  γ = {fmt(gm, 4)}: {trial(gamma=gm)}")

A larger $\alpha$ gives a more even load, but at $\alpha = 1$ the routing carries less information about the digit
than at $\alpha = 0.1$, like the demo's 10-epoch × 3-seed results in section 9. With $\gamma = 0.0001$, the bias is
too slow: after 429 steps each $b_i$ can have moved by at most $0.0429$, while the winning expert's score is already
far ahead, so the model has time to collapse first.

### Top-1 with a normalized gate

Why does Switch (top-1) keep the raw gate $g = s_i$? Try normalizing it: with $k = 1$, $g = s_i / s_i = 1$ for every
token.

In [ ]:
P_r, _, _ = train(0, k=1, gate="renorm", lr=3e-3)
moved = np.abs(P_r["Wr"] - init_moe(0)["Wr"]).max()
print(
    f"top-1, normalized gate: W_r changed by at most {fmt(moved, 6)} after 429 steps · {summary(P_r, None, 1, 'renorm')}"
)
P_raw, _, _ = live[("e8k1_none", 0)]  # same seed, raw gate, from section 5
print(f"top-1, raw gate (section 5):            · {summary(P_raw, None, 1, 'raw')}")
assert moved == 0.0

$W_r$ doesn't change by a single bit. The gate is always 1, so $\partial g/\partial s = 0$: the router receives no
gradient from the task, and it is just a fixed random projection of $h$, like the unlearned router of section 9.
Because there is no self-reinforcing loop through $W_r$, it doesn't collapse as badly as with the raw gate (second
line: 6 dead experts), but it doesn't learn to split the work by content either.
The gradient reaches the router *only* through the value of the gate; Shazeer et al. once thought $k \ge 2$ was
needed to compare two experts, while Switch showed that $k = 1$ is enough, as long as the output is multiplied by
$s_i$.

## 12 · Limits and next steps

MoE makes the FFN layer cheaper per token, but doesn't touch attention. Each new token still has to reread the keys
and values of every token before it (Lesson 17), so as the context grows longer, attention and its KV cache become
the most expensive part. Lesson 25 asks: can we replace attention with a fixed-size state, like an RNN, and still
remember enough?

MoE doesn't reduce memory either: every expert has to be loaded in advance (87.0 GiB for Mixtral, 641 GiB for
DeepSeek-V3), and moving tokens back and forth between the GPUs that hold the experts is a separate cost that this
notebook doesn't measure.

## Summary
- The MoE layer: a linear router $s = \operatorname{softmax}(hW_r)$, top-k selection, a normalized gate,
  $u = h + \sum g_i \mathrm{FFN}_i(h)$. An expert that is not selected doesn't run and receives no gradient.
- Mixtral 8x7B: 46.7 billion parameters (not 56), 12.9 billion per token; top-2 costs 1.78 times as much as top-1;
  87.0 GiB of weights. DeepSeek-V3: 671 / 37.6 billion. Computed from config.json, matching the checkpoint down to
  the last parameter.
- Without balancing, the router collapses: 5–7 of the 8 experts are dead after 43 steps, and they don't recover.
- The auxiliary loss $\alpha N \sum f_i P_i$ (Switch) or the selection-only bias (DeepSeek-V3) keeps every expert
  alive; too large an $\alpha$ makes the routing lose information.
- Capacity $C = \lceil \mathrm{cf} \cdot nk/N \rceil$: at cf = 1, the collapsed model drops 64–88% of its assignments
  (86.1–96.4% correct), the balanced model 19–20% (97.1–97.3%): dropped tokens go through the residual path, and on
  MNIST its experts contribute very little. Uniform random routing still drops 9%.
- The experts group digits that look alike (3–5 in 11/12 runs, 4–9 in 9/12); they don't learn "topics".
- On MNIST, MoE beats a dense network by only a few tenths of a point with a residual path, and loses without one.

## Review questions
1. Why does Mixtral 8x7B have 46.7 billion parameters rather than 8 × 7 = 56 billion?
2. Why does top-2 cost 1.78 times as much as top-1, rather than 2 times?
3. Why does a router that isn't balanced collapse on its own, and so early?
4. How does DeepSeek-V3's bias $b_i$ balance the load without pulling the model away from its main objective?
5. The average load is exactly equal to the capacity (cf = 1), so why are some tokens still dropped? Where do the
   dropped tokens go?
6. With top-1 and a normalized gate, what does the router learn?

<details><summary>Hints and answers</summary>

1. Only the FFN layers are multiplied 8 times; attention, the embeddings and normalization (about 1.6 billion) are
   shared.
2. The shared part runs once whatever $k$ is; only the expert part doubles.
3. A selected expert receives gradients and learns faster, so its gate grows and it gets selected even more; an expert
   that isn't selected learns nothing, so it never becomes attractive again. At the start all the scores are nearly
   equal, so a small difference is enough to set it off, and the load is already uneven in the very first batch
   (section 5).
4. $b_i$ is added to the scores only when selecting the top k, not to the gate, so it is not in the loss and receives
   no gradient. After each step, $b_i$ goes up by $\gamma$ if the expert is underloaded, and down by $\gamma$ if it is
   overloaded.
5. By chance, some expert receives more than $C$ in a batch and the excess is dropped, while other experts leave slots
   empty. A dropped token doesn't go through that expert at that layer; it continues along the residual path
   (Switch).
6. Nothing: the gate is always 1, so $W_r$ receives no gradient and stays as it was at initialization. The choices
   still change as the trunk learns, but the router is only a fixed random projection of $h$ (section 11).
</details>

## Further reading
- R. Jacobs, M. Jordan, S. Nowlan & G. Hinton (1991), [Adaptive Mixtures of Local Experts](https://www.cs.toronto.edu/~hinton/absps/jjnh91.pdf).
- N. Shazeer et al. (2017), [Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer](https://arxiv.org/abs/1701.06538):
  noisy top-k, and the router's self-reinforcing loop.
- W. Fedus, B. Zoph & N. Shazeer (2021), [Switch Transformers](https://arxiv.org/abs/2101.03961): top-1, capacity,
  the auxiliary loss $\alpha N \sum f_i P_i$.
- A. Jiang et al. (2024), [Mixtral of Experts](https://arxiv.org/abs/2401.04088): an analysis of routing by domain and
  by syntax.
- D. Dai et al. (2024), [DeepSeekMoE](https://arxiv.org/abs/2401.06066): small experts and shared experts.
- L. Wang et al. (2024), [Auxiliary-Loss-Free Load Balancing](https://arxiv.org/abs/2408.15664) and
  DeepSeek-AI (2024), [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437).
- N. Muennighoff et al. (2024), [OLMoE](https://arxiv.org/abs/2409.02060): expert specialization, measured carefully.
- Hugging Face (2023), [Mixture of Experts Explained](https://huggingface.co/blog/moe).

**Next lesson:** [Lesson 25 · RNNs are back: linear attention & Mamba](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rnn-tro-lai/):
replace attention with a fixed-size state.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")